# Who Pays What? (starter)

Read `README.md` first. This notebook loads the data, trains a plain baseline on the numeric columns, and scores it. Your job: beat it.

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score

TARGET = "ukt_bracket"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
print(train.shape, test.shape)
train.head()

## A quick look at the messy text

In [ ]:
for col in ["roof_material", "wall_material", "floor_material"]:
    print(col, "->", train[col].nunique(), "different answers")
    print(train[col].value_counts().head(8).to_dict(), "\n")

## Scoring helper
Use it only at the end. For decisions, use cross-validation on `train`.

In [ ]:
def score(pred):
    """pred: array of brackets (0-7), same order as test.csv"""
    labels = pd.read_csv("data/test_labels.csv").set_index("id").loc[test["id"], TARGET].values
    f1 = f1_score(labels, pred, average="macro")
    print(f"test macro-F1 = {f1:.4f}   accuracy = {accuracy_score(labels, pred):.4f}")
    return f1

## Baseline: XGBoost on the numeric columns

In [ ]:
from xgboost import XGBClassifier

numeric = [c for c in test.columns if test[c].dtype != object and c != "id"]
model = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=42)
model.fit(train[numeric], train[TARGET])
pred = model.predict(test[numeric])

## Your predictions
One row per test form, columns `id, ukt_bracket`.

In [ ]:
submission = pd.DataFrame({"id": test["id"], TARGET: pred})
submission.head()

In [ ]:
score(submission[TARGET])

## Ideas to try
- Many text columns hide a small number of real answers behind many spellings. How could you group spellings that are *almost* the same?
- Some categories have an order (income ranges, education levels).
- Look at bracket 0 in `train`. What do those students have in common?
- Macro-F1 punishes ignoring a rare bracket.